# 全连接网络（PyTorch 高层版）

与手写版（C/W1/b1/W2/b2）数学等价，只是换成 `nn.Embedding` / `nn.Linear` / 优化器。手写版负责理解机制，这版是最终落地形态。

## 1. 数据准备（和第 3 课完全一样，block_size=3）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

words = open('prenoms.txt', encoding='utf-8').read().splitlines()
words = [w for w in words if w.strip()]
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}; stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
V = len(stoi)

block_size = 3
X, Y = [], []
for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context); Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X); Y = torch.tensor(Y)

n = len(X)
train_set, val_set, test_set = random_split(
    TensorDataset(X, Y),
    [int(0.8*n), int(0.1*n), n - int(0.8*n) - int(0.1*n)]
)
train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)
print(X.shape, Y.shape)

## 2. 模型：nn.Embedding + nn.Sequential

对照表：`nn.Embedding` = 手写 C；`nn.Linear(30, 200)` = W1+b1；`nn.Tanh()` = tanh；`nn.Linear(200, 46)` = W2+b2。

In [ ]:
class CharLM(nn.Module):
    def __init__(self, vocab_size, block_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)   # 对应手写 C
        self.net = nn.Sequential(
            nn.Linear(block_size * embed_dim, hidden_dim),     # 对应 W1 + b1
            nn.Tanh(),
            nn.Linear(hidden_dim, vocab_size),                 # 对应 W2 + b2
        )

    def forward(self, x):                # x: (N, block_size)
        emb = self.embedding(x)          # (N, 3, 10)
        emb = emb.view(x.size(0), -1)    # (N, 30)
        return self.net(emb)             # (N, 46) logits

model = CharLM(V, block_size, embed_dim=10, hidden_dim=200)
print('参数量:', sum(p.numel() for p in model.parameters()))   # 15906

## 3. 训练：优化器 + lr 衰减

`optimizer.zero_grad(); loss.backward(); optimizer.step()` 替代手写的 `p.grad=None; p.data -= lr*p.grad`。

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

for epoch in range(100):
    model.train()
    loss_epoch = 0.0
    for x, y in train_loader:
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_epoch += loss.item()
    loss_epoch /= len(train_loader)

    model.eval()
    loss_val = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            loss_val += F.cross_entropy(model(x), y).item()
    loss_val /= len(val_loader)

    if epoch == 50:
        for g in optimizer.param_groups:
            g['lr'] = 0.02              # 50 epoch 后 lr 衰减 10 倍

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | train {loss_epoch:.3f} | val {loss_val:.3f}')

## 4. 测试集评估

In [ ]:
model.eval()
loss_test = 0.0
with torch.no_grad():
    for x, y in test_loader:
        loss_test += F.cross_entropy(model(x), y).item()
loss_test /= len(test_loader)
print('测试 loss:', round(loss_test, 4))

## 5. 生成名字

In [ ]:
model.eval()
def gen_name():
    context = [0] * block_size
    out = []
    with torch.no_grad():
        while True:
            logits = model(torch.tensor([context]))
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, 1).item()
            context = context[1:] + [ix]
            out.append(itos[ix])
            if ix == 0: break
    return ''.join(out)

for _ in range(20):
    print(gen_name())